# 时域和门控功能

## 简介

本笔记本演示了如何使用 [scikit-rf](http://scikit-rf.org) 进行时域分析和门控。首先给出一个快速示例，然后进行更详细的解释。散射参数是在频域中测量的，但如果需要，也可以在时域中进行分析。在许多情况下，测量并不延伸到直流 (DC)。这意味着时域变换是不完整的，但即便如此，它仍然非常有用。时域分析的主要应用之一是使用 *门控* 来隔离空间中的单个响应。有关时域分析细节的更多信息，请参见 [1]。参考文献：* [1] 凯恩科技 - 使用网络分析仪进行时域分析 - 应用笔记 [pdf](https://www.keysight.com/us/en/assets/7018-01451/application-notes/5989-5723.pdf)

## 快速示例

In [ ]:
import matplotlib.pyplot as plt

import skrf as rf

%matplotlib inline
rf.stylely()

# load data for the waveguide to CPW probe
probe = rf.Network('../metrology/oneport_tiered_calibration/probe.s2p')

# we will focus on s11
s11 = probe.s11

#  time-gate the first largest reflection
s11_gated = s11.time_gate(center=0, span=.2, t_unit='ns')
s11_gated.name='gated probe'

# plot frequency and time-domain s-parameters
plt.figure(figsize=(8,4))
plt.subplot(121)
s11.plot_s_db()
s11_gated.plot_s_db()
plt.title('Frequency Domain')

plt.subplot(122)
s11.plot_s_db_time()
s11_gated.plot_s_db_time()
plt.title('Time Domain')
plt.tight_layout()

## 时域分析

在本示例中，DUT（待测设备）是一个波导到共面波导（CPW）探头，其测量结果已在[此其他示例](../metrology/One Port Tiered Calibration.ipynb)中给出。

In [ ]:
# load data for the waveguide to CPW probe
probe = rf.Network('../metrology/oneport_tiered_calibration/probe.s2p')
probe

请注意，scikit-rf 中有两个时域绘图函数：* `Network.plot_s_db_time()`* `Network.plot_s_time_db()`两者的区别在于，前者 `plot_s_db_time()` 在绘图之前使用窗函数来提高脉冲分辨率。窗函数稍后会进行讨论，但现在我们只需使用 `plot_s_db_time()`。在频率域和时域中同时绘制探头的所有四个散射参数。

In [ ]:
# plot frequency and time-domain s-parameters
plt.figure(figsize=(8,4))
plt.subplot(121)
probe.plot_s_db()
plt.title('Frequency Domain')
plt.subplot(122)
probe.plot_s_db_time()
plt.title('Time Domain')
plt.tight_layout()

关注波导端口的反射系数 (s11)，可以看到存在干涉现象。

In [ ]:
probe.plot_s_db(0,0)
plt.title('Reflection Coefficient From \nWaveguide Port')

这种波纹是多个离散反射存在的证据。在时域中绘制 s11 图可以让我们看到这些反射发生的位置，或者说*发生的时间*。

In [ ]:
probe_s11 = probe.s11
probe_s11.plot_s_db_time(0,0)
plt.title('Reflection Coefficient From \nWaveguide Port, Time Domain')
plt.ylim(-100,0)

从这个图中我们可以看到两个主要的反射信号：* 一个在 $t=0$ ns 处（测试端口）* 另一个在 $t=0.2$ ns 处（谁知道呢？）。

## 隔离感兴趣的反射信号

为了隔离波导端口的反射，我们可以使用时域门控。这可以通过使用 `Network.time_gate()` 方法来实现，并为其提供合适的中心时间和门控范围（以纳秒为单位）。为了观察门控效果，我们将比较原始响应和门控后的响应。

In [ ]:
probe_s11_gated = probe_s11.time_gate(center=0, span=.2, t_unit='ns')
probe_s11_gated.name='gated probe'

s11.plot_s_db_time()
s11_gated.plot_s_db_time()

接下来，在频域中比较这两个响应，以查看栅极的影响。

In [ ]:
s11.plot_s_db()
s11_gated.plot_s_db()

### 自动门控`skrf` 中的时域门控方法具有自动门控功能，也可以用于门控最大的反射信号。如果没有提供门控参数，`time_gate()` 函数将执行以下操作：1. 找到两个最大的峰值* 将门控中心设置为最高的峰值* 将跨度设置为两个最高峰值之间的距离您可能希望在时域中绘制门控后的网络，以查看确定的门控形状。

In [ ]:
plt.title('Waveguide Interface of Probe')
s11.plot_s_db(label='original')
s11.time_gate().plot_s_db(label='autogated') #autogate on the fly

可以尝试看看自动门在另一个探头接口上的表现如何。

In [ ]:
plt.title('Other Interface of Probe')
probe.s22.plot_s_db()
probe.s22.time_gate().plot_s_db()


## 确定距离

为了使时域分析能够作为一种诊断工具，我们需要将 x 轴转换为距离。这需要了解器件中的传播速度。**skrf** 在 [skrf.media](../../api/media/index.rst) 模块中提供了一些传输线模型，可用于此目的。**但是……**对于色散介质，例如矩形波导，相位速度是频率的函数，因此将时间转换为距离并不简单。作为一种近似方法，您可以将 x 轴归一化为光速。或者，您可以模拟一个已知的器件，并比较两个时域响应。这使您可以为坐标轴赋予量化的意义。例如，您可以创建一个理想的延迟负载，如下所示。请注意：一个大的脉冲 *之后* 的响应的幅度没有有意义的单位。

In [ ]:
from skrf.constants import mil
from skrf.media import RectangularWaveguide

# create a rectangular waveguide media to generate a theoretical network
wr1p5 = RectangularWaveguide(frequency=probe.frequency,
                             a=15*mil,z0_override=1)

# create an ideal delayed load, parameters are adjusted until the
# theoretical response agrees with the measurement
theory = wr1p5.delay_load(Gamma0=rf.mathFunctions.db_2_mag(-20),
                          d=2.4, unit='cm')


probe.plot_s_db_time(0,0, label = 'Measurement')
theory.plot_s_db_time(label='-20dB @ 2.4cm from test-port')
plt.ylim(-100,0)

这个图表展示了一些重要的要点：* 理论上的延迟负载在时域中并不是一个完美的脉冲。这是由于波导中的色散造成的。* 时域中幅值的峰值与指定的峰值并不完全相同，这也是由于色散（以及窗口函数）造成的。

## 窗口函数到底是什么？

`'plot_s_db_time()'` 函数执行以下几个操作。1.  对散射参数进行窗口处理。    *   转换为时域    *   提取幅度分量，转换为 dB    *   计算时轴上的散射参数    *   绘制图形关于步骤 1，需要说明的是：**窗口处理**。FFT（快速傅里叶变换）使用一系列周期信号（正弦波）来表示信号。如果您的频率响应不是周期性的（通常情况下并非如此），那么进行 FFT 变换会在时域结果中引入伪影。为了最大限度地减少这些影响，会对频率响应进行*窗口处理*。这通过逐渐减小带边来使频率响应更具周期性。窗口处理仅用于改善图形的外观，它不会影响原始网络。在 skrf 中，可以使用 `'windowed()'` 函数显式地执行此操作。默认情况下，此函数使用汉明窗口，但可以通过参数进行调整。窗口处理的结果如下所示。

In [ ]:
probe_w = probe.windowed()
probe.plot_s_db(0,0, label = 'Original')
probe_w.plot_s_db(0,0, label = 'Windowed')


通过比较这两个时域绘图函数，我们可以看到使用窗口函数和不使用窗口函数的区别。

In [ ]:
probe.plot_s_time_db(0,0, label = 'Original')
probe_w.plot_s_time_db(0,0, label = 'Windowed')